# 🚀 PPO for Language Models (LLMs) using Hugging Face Transformers

This notebook adapts a classic PPO implementation for use in training a language model, such as GPT-2, using reinforcement learning techniques like PPO. 
We use Hugging Face's `trl` library to train a language model from prompt-response-reward triplets.

Perfect for projects like:
- Reinforcement Learning from Human Feedback (RLHF)
- Reward model fine-tuning
- Prompt optimization

---


In [ ]:
!pip install trl transformers datasets accelerate -q

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import PPOTrainer, PPOConfig
from datasets import Dataset
import torch


In [ ]:

# Load model and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)


In [ ]:

# PPO configuration
config = PPOConfig(
    model_name=model_name,
    batch_size=2,
    learning_rate=1e-5,
    log_with=None,
)

# Example prompt dataset
prompts = [
    "Explain the theory of relativity in simple terms.",
    "How do airplanes fly?",
    "What is the capital of France?",
    "Why is the sky blue?"
]

dataset = Dataset.from_dict({"prompt": prompts})


In [ ]:

def tokenize(sample):
    return tokenizer(sample["prompt"], return_tensors="pt", padding=True, truncation=True)
dataset = dataset.map(tokenize, batched=False)


In [ ]:

def reward_fn(prompt, response):
    # Toy reward: favor shorter responses
    return -abs(len(response.split()) - 5)


In [ ]:

ppo_trainer = PPOTrainer(config=config, model=model, tokenizer=tokenizer, dataset=dataset)


In [ ]:

for epoch in range(2):  # Short run for demo
    print(f"Epoch {epoch + 1}")
    for sample in dataset:
        query_tensors = sample["input_ids"]
        response_tensors = ppo_trainer.generate(query_tensors, max_new_tokens=20)
        responses = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)
        queries = tokenizer.batch_decode(query_tensors, skip_special_tokens=True)
        rewards = [reward_fn(q, r) for q, r in zip(queries, responses)]
        ppo_trainer.step(query_tensors, response_tensors, rewards)
        print(f"Prompt: {queries[0]}
Response: {responses[0]}
Reward: {rewards[0]}")
        print("-" * 60)
